# fetch 재연결과 오류 이벤트

POST SSE에서 클라이언트가 마지막 이벤트 ID를 보관해 다시 보내고, 스트림 도중 오류를 이벤트로 받는 과정을 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

첫 요청의 `id`를 읽어 다음 요청의 `Last-Event-ID` 헤더로 보냅니다.

In [2]:
import re
from fastapi.testclient import TestClient
from agent_service.app import app
client = TestClient(app)
auth = {'Authorization': 'Bearer local-demo-token'}
body = {'question': '재연결 확인 질문', 'thread_id': 'resume-1'}
first = client.post('/v1/chat/stream', headers=auth, json=body)
ids = re.findall(r'id: ([^\n]+)', first.text)
print(ids[:2])
assert ids[0] == 'resume-1:0'

['resume-1:0', 'resume-1:1']


## 재연결 위치 관찰

두 번째 요청은 마지막으로 처리한 ID 다음 토큰부터 전달합니다.

In [3]:
resume_headers = {**auth, 'Last-Event-ID': ids[0]}
resumed = client.post('/v1/chat/stream', headers=resume_headers, json=body)
resumed_ids = re.findall(r'id: ([^\n]+)', resumed.text)
print(resumed_ids[0])
assert resumed_ids[0] == 'resume-1:1'

resume-1:1


## 실패 재현

스트림이 이미 200으로 시작한 뒤 Agent가 실패합니다.

In [4]:
failed = client.post('/v1/chat/stream', headers=auth,
    json={'question': '오류', 'thread_id': 'error-1'})
print(failed.status_code)
print(failed.text.strip())
assert failed.status_code == 200

Agent SSE stream failed
Traceback (most recent call last):
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/agent_service/app.py", line 110, in chat_stream
    async for chunk in runtime.astream(
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/.venv/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3440, in astream
    async for _ in runner.atick(
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/.venv/lib/python3.12/site-packages/langgraph/pregel/_runner.py", line 557, in atick
    _panic_or_proceed(
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/.venv/lib/python3.12/site-packages/langgraph/pregel/_runner.py", line 687, in _panic_or_proceed
    raise exc
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/.venv/lib/python3.12/site-packages/langgraph/pregel/_retry.py", line 744, in arun_with_retry
    return await task.proc

200
event: error
data: {"code": "stream_failed", "message": "Agent \uc2a4\ud2b8\ub9bc\uc774 \uc911\ub2e8\ub418\uc5c8\uc2b5\ub2c8\ub2e4"}


## 수정

클라이언트는 HTTP 상태만 보지 않고 `event: error`와 오류 코드를 확인합니다.

In [5]:
has_error = 'event: error' in failed.text
has_code = 'stream_failed' in failed.text
print(has_error, has_code)
assert has_error and has_code

True True
